# VERSION B - LANGCHAIN

## Orchestration

This version uses LangChain for agent orchestration.

In [1]:
%pip install -U openai langchain langchain-core langchain-openai 

   ---------------------------------------- 0.0/571.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.8 kB ? eta -:--:--
   ---------------------------------- --- 524.3/571.8 kB 598.5 kB/s eta 0:00:01
   ---------------------------------------- 571.8/571.8 kB 439.8 kB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.2
    Uninstalling langchain-core-1.6.2:
      Successfully uninstalled langchain-core-1.6.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json
import requests
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(find_dotenv(usecwd=True))
MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL,temperature=0)

## Weather Tool

In [3]:
@tool
def get_destination_weather(city: str, days_ahead: int = 7) -> dict:
    """
    Get the weather forecast for a destination city.
    Returns an error dictionary if the city cannot be resolved.
    """

    try:
        geo_url = (
            "https://geocoding-api.open-meteo.com/v1/search"
        )

        geo_params = {
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }

        geo_response = requests.get(
            geo_url,
            params=geo_params
        )

        geo_data = geo_response.json()

        if not geo_data.get("results"):
            raise ValueError(
                f"Could not resolve city: {city}"
            )

        location = geo_data["results"][0]

        latitude = location["latitude"]
        longitude = location["longitude"]
        city_name = location["name"]

        weather_url = (
            "https://api.open-meteo.com/v1/forecast"
        )

        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "daily": (
                "temperature_2m_min,"
                "temperature_2m_max,"
                "precipitation_probability_max"
            ),
            "forecast_days": 16,
            "timezone": "auto"
        }

        weather_response = requests.get(
            weather_url,
            params=weather_params
        )

        weather_data = weather_response.json()

        if "daily" not in weather_data:
            raise ValueError(
                f"Could not get weather data: {weather_data}"
            )

        daily = weather_data["daily"]

        return {
            "city": city_name,
            "date": daily["time"][days_ahead],
            "min_temp_c": daily[
                "temperature_2m_min"
            ][days_ahead],
            "max_temp_c": daily[
                "temperature_2m_max"
            ][days_ahead],
            "rain_probability": daily[
                "precipitation_probability_max"
            ][days_ahead]
        }

    except Exception as e:
        return {
            "error": f"{type(e).__name__}: {e}"
        }

## Currency Tool

In [4]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """
    Convert money from one currency to another.
    """

    url = ("https://api.frankfurter.app/latest")

    params = {
        "amount": amount,
        "from": from_currency.upper(),
        "to": to_currency.upper() 
    }

    response = requests.get(url, params=params)

    data = response.json()

    if "rates" not in data:
        raise ValueError(f"Could not convert {from_currency} to {to_currency}")

    converted_amount = data["rates"][to_currency.upper()]
    return {
        "amount": amount,
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "converted_amount": converted_amount
    }    

## Trip Cost Tool

In [5]:
@tool
def estimate_trip_cost(city: str, days: int, travel_style: str = "budget") -> dict:
    """
    Estimate trip cost in INR based on city, number of days, and travel style.
    """

    city_tiers = {
        "ooty": 2500,
        "manali": 3000,
        "bangkok": 4000,
        "bali": 5000
    }

    style_multiplier = {
        "budget": 1.0,
        "mid-range": 1.7,
        "luxury": 3.0
    }

    city_key = city.lower()
    daily_cost = city_tiers.get(city_key, 3500)
    multiplier = style_multiplier.get(travel_style.lower(), 1.0)
    total_cost = (daily_cost * days * multiplier)
    result = {
        "city": city,
        "days": days,
        "travel_style": travel_style,
        "estimated_cost_inr": round(total_cost, 2)
    }

    if city_key not in city_tiers:
        result["note"] = ("Unknown city. Default cost tier used.")

    return result

## System Prompt

In [6]:
Tripbuddy_system = """
You are a Tripbuddy, a helpful travel planning assistant.
Use the available tools when needed.
Use conversation memory for follow-up questions.
Handle tool errors clearly and give concise answers.
"""

## Tools + Memory + Agent

In [7]:
trip_tools = [
    get_destination_weather,
    convert_currency,
    estimate_trip_cost
]

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=trip_tools,
    system_prompt=Tripbuddy_system,
    checkpointer=memory
)

## Run Function

In [8]:
def run_tripbuddy(user_message, thread_id="default"):

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print("USER:")
    print(user_message)
    print()

    for chunk in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ]
        },
        config=config,
        stream_mode="updates"
    ):

        if "model" in chunk:

            messages = chunk["model"].get("messages", [])

            for message in messages:

                if message.tool_calls:

                    for tool_call in message.tool_calls:

                        print("MODEL:")
                        print(
                            f"Calling tool: "
                            f"{tool_call['name']}"
                        )

                        print(
                            f"Arguments: "
                            f"{tool_call['args']}"
                        )

                        print()

                elif message.content:

                    print("FINAL ANSWER:")
                    print(message.content)
                    print()

        if "tools" in chunk:

            messages = chunk["tools"].get("messages", [])

            for message in messages:

                print("TOOL RESULT:")
                print(message.content)
                print()

## Query 1 — Weather

What's the weather in Ooty tomorrow?

In [ ]:
run_tripbuddy(
    "What's the weather in Ooty tomorrow?",
    thread_id="query1"
)

USER:
What's the weather in Ooty tomorrow?

FINAL ANSWER:
I encountered an issue while trying to fetch the weather for Ooty. Unfortunately, I can't provide the weather information at the moment. You might want to check a weather website or app for the latest updates.



## Query 2 — Weather and Trip Cost

I'm going to Manali for 4 days next week —
should I pack for rain, and roughly what will it cost?

In [11]:
run_tripbuddy(
    "I'm going to Manali for 4 days next week — should I pack for rain, and roughly what will it cost?",
    thread_id="manali_trip"
)

USER:
I'm going to Manali for 4 days next week — should I pack for rain, and roughly what will it cost?

MODEL:
Calling tool: get_destination_weather
Arguments: {'city': 'Manali', 'days_ahead': 7}

MODEL:
Calling tool: estimate_trip_cost
Arguments: {'city': 'Manali', 'days': 4}

TOOL RESULT:
{"city": "Manali", "days": 4, "travel_style": "budget", "estimated_cost_inr": 12000.0}

TOOL RESULT:
{"city": "Manali", "date": "2026-09-19", "min_temp_c": 24.2, "max_temp_c": 33.1, "rain_probability": 89}

FINAL ANSWER:
Next week in Manali, there's a high probability of rain (89%), so it's a good idea to pack for wet weather.

As for the cost, a budget trip for 4 days in Manali is estimated to be around ₹12,000.



## Query 3 — Currency Conversion

Convert 500 USD to INR for my Bali trip.

In [ ]:
run_tripbuddy(
    "Convert 500 USD to INR for my Bali trip.",
    thread_id="query3"
)

USER:
Convert 500 USD to INR for my Bali trip.

FINAL ANSWER:
500 USD is approximately 47,720 INR for your Bali trip.



## Query 4 — All Three Tools

Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?

In [ ]:
run_tripbuddy(
    "Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?",
    thread_id="bangkok_trip"
)

USER:
Planning a 6-day trip to Bangkok. What's the weather, what will it cost in INR, and if I already have 20,000 THB, how much is that in INR?

MODEL:
Calling tool: get_destination_weather
Arguments: {'city': 'Bangkok'}

MODEL:
Calling tool: estimate_trip_cost
Arguments: {'city': 'Bangkok', 'days': 6}

MODEL:
Calling tool: convert_currency
Arguments: {'amount': 20000, 'from_currency': 'THB', 'to_currency': 'INR'}

TOOL RESULT:
{"city": "Bangkok", "days": 6, "travel_style": "budget", "estimated_cost_inr": 24000.0}

TOOL RESULT:
{"amount": 20000.0, "from_currency": "THB", "to_currency": "INR", "converted_amount": 57852}

TOOL RESULT:
{"city": "Bangkok", "date": "2026-09-18", "min_temp_c": 25.4, "max_temp_c": 27.4, "rain_probability": 98}

FINAL ANSWER:
Here's the information for your 6-day trip to Bangkok:

### Weather
- **Date**: September 18, 2026
- **Min Temperature**: 25.4°C
- **Max Temperature**: 27.4°C
- **Rain Probability**: 98%

### Estimated Trip Cost (Budget Style)
- **Estima

## Query 5 — Invalid City Error Handling

What's the weather in Zzyxville next week?

In [ ]:
run_tripbuddy(
    "What's the weather in Zzyxville next week?",
    thread_id="query5"
)

USER:
What's the weather in Zzyxville next week?

MODEL:
Calling tool: get_destination_weather
Arguments: {'city': 'Zzyxville'}

TOOL RESULT:
{"error": "ValueError: Could not resolve city: Zzyxville"}

FINAL ANSWER:
I still can't find any information for Zzyxville. It might not be a recognized location. If you have another city in mind, please let me know!



## Query 6 — Follow-Up Using Memory

Make that a mid-range trip instead.

In [ ]:
run_tripbuddy(
    "Make that a mid-range trip instead.",
    thread_id="bangkok_trip"
)

USER:
Make that a mid-range trip instead.

MODEL:
Calling tool: estimate_trip_cost
Arguments: {'city': 'Bangkok', 'days': 6, 'travel_style': 'mid-range'}

TOOL RESULT:
{"city": "Bangkok", "days": 6, "travel_style": "mid-range", "estimated_cost_inr": 40800.0}

FINAL ANSWER:
For a mid-range trip to Bangkok for 6 days, the estimated cost would be **₹40,800**.

If you have any more questions or need further assistance, just let me know!

